# Volume 1. What Is an Assembly?

Question: what changes as sparse activity stabilizes, and what changes again when two traces are bound into a new one?

This first notebook uses a tiny concept-binding scene. A `red_cue` projects into a `COLOR` area, a `triangle_cue` projects into a `SHAPE` area, and `merge_trace` lets those two source assemblies drive a new `OBJECT` assembly.

Nothing here recognizes color, geometry, or images. The labels are handles for routing and inspection. The plots are index maps: useful for seeing sparse winners and winner turnover, not cortical anatomy.

In [ ]:
import copy

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

from neural_assemblies.assembly_calculus import Assembly, chance_overlap, overlap
from neural_assemblies.assembly_calculus.trace import (
    merge_trace,
    project_trace,
    reciprocal_project_trace,
)
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import (
    animate_assembly_trace,
    plot_assemblies,
    plot_binding_story,
    plot_response_overlap,
    plot_trace_metrics,
)

The whole experiment has one shape: form two source assemblies, watch the target activity settle over rounds, and then ask whether either source can later recover the merged target trace.

In [ ]:
flow_fig = plot_binding_story(
    stimulus_labels=["red cue\nexternal input", "triangle cue\nexternal input"],
    source_labels=["COLOR area\nred assembly", "SHAPE area\ntriangle assembly"],
    target_label="OBJECT area\nred-triangle assembly",
    title="Binding two perceptual cues into one object assembly",
    caption="project each cue, then merge the two source assemblies into a new target trace",
)
plt.show()
plt.close(flow_fig)

Set up the bench. `N` is the neuron budget in each area, `K` is the number of active winners per assembly, and `BETA` controls Hebbian strengthening after co-activation.

In [ ]:
N = 5_000
K = 80
BETA = 0.08
PROJECTION_ROUNDS = 8
MERGE_ROUNDS = 5
SOURCE_RESPONSE_ROUNDS = 6
SEED = 42

STIMULI = {
    "red cue": "red_cue",
    "triangle cue": "triangle_cue",
}
AREAS = {
    "color": "COLOR",
    "shape": "SHAPE",
    "object": "OBJECT",
}
PALETTE = {
    "red": "#d64f3a",
    "triangle": "#3169b3",
    "object": "#2f8f65",
    "baseline": "#6f6f6f",
}

brain = Brain(p=0.05, save_winners=True, seed=SEED, engine="numpy_sparse")

for stimulus in STIMULI.values():
    brain.add_stimulus(stimulus, K)

for area in AREAS.values():
    brain.add_area(area, n=N, k=K, beta=BETA)

Start with a cast of characters. This makes the story labels, routing choices, and inspection surfaces explicit before any activity is generated.

In [ ]:
cast = pd.DataFrame(
    [
        {
            "handle": "red_cue",
            "kind": "stimulus",
            "role": "external drive for the red cue",
            "where it writes": "COLOR",
            "budget": K,
        },
        {
            "handle": "triangle_cue",
            "kind": "stimulus",
            "role": "external drive for the triangle cue",
            "where it writes": "SHAPE",
            "budget": K,
        },
        {
            "handle": "COLOR",
            "kind": "area",
            "role": "stores the red source assembly",
            "where it writes": "OBJECT during merge",
            "budget": f"N={N}, K={K}",
        },
        {
            "handle": "SHAPE",
            "kind": "area",
            "role": "stores the triangle source assembly",
            "where it writes": "OBJECT during merge",
            "budget": f"N={N}, K={K}",
        },
        {
            "handle": "OBJECT",
            "kind": "area",
            "role": "receives joint input from COLOR and SHAPE",
            "where it writes": "target assembly",
            "budget": f"N={N}, K={K}",
        },
    ]
)
cast

The package trace helpers record one immutable assembly snapshot per round. The local helper below only formats final snapshots into compact inspection tables.

In [ ]:
def assembly_table(named_assemblies, n):
    """Return size, density, and sample winner indices for assembly snapshots."""
    rows = []
    for label, assembly in named_assemblies:
        rows.append(
            {
                "assembly": label,
                "area": assembly.area,
                "winner_count": len(assembly),
                "density": f"{len(assembly) / n:.2%}",
                "sample_winners": assembly.winners[:12].tolist(),
            }
        )
    return pd.DataFrame(rows)

Projection is a process, not a teleport. Round 1 is cue-only. Later rounds combine cue input with recurrence, so the active winner set can turn over before it stabilizes.

In [ ]:
red_trace = project_trace(
    brain,
    STIMULI["red cue"],
    AREAS["color"],
    rounds=PROJECTION_ROUNDS,
)
triangle_trace = project_trace(
    brain,
    STIMULI["triangle cue"],
    AREAS["shape"],
    rounds=PROJECTION_ROUNDS,
)

red_assembly = red_trace.final
triangle_assembly = triangle_trace.final

pd.DataFrame(red_trace.to_records())[
    ["round", "drive", "winner_count", "ever_fired", "new_winners", "overlap_prev"]
]

Watch the `COLOR` trace settle. The number of active winners stays fixed at `K`, but the identity of those winners changes until recurrence stabilizes the trace.

In [ ]:
red_anim_fig, red_animation = animate_assembly_trace(
    red_trace,
    N,
    title="COLOR projection dynamics",
    color=PALETTE["red"],
    interval=650,
)
red_animation_html = HTML(red_animation.to_jshtml())
plt.close(red_anim_fig)
red_animation_html

The same dynamics are easier to quantify as two small curves: consecutive overlap rises as the trace settles, while new winners fall as the active set stops turning over.

In [ ]:
metrics_fig, metrics_axes = plot_trace_metrics(
    red_trace,
    title="COLOR projection stabilization",
    color=PALETTE["red"],
)
plt.show()
plt.close(metrics_fig)

The source assemblies are sparse final states from those traces. The sample winner lists are concrete handles for later comparisons.

In [ ]:
assembly_table(
    [
        ("red cue -> COLOR", red_assembly),
        ("triangle cue -> SHAPE", triangle_assembly),
    ],
    N,
)

In [ ]:
fig, axes = plot_assemblies(
    [red_assembly, triangle_assembly],
    n=N,
    titles=["COLOR: red cue", "SHAPE: triangle cue"],
    subtitles=[
        f"{len(red_assembly)} winners | {len(red_assembly) / N:.2%} density",
        f"{len(triangle_assembly)} winners | {len(triangle_assembly) / N:.2%} density",
    ],
    colors=[PALETTE["red"], PALETTE["triangle"]],
    marker_size=18,
)
fig.suptitle("Source assemblies after projection", y=1.06, fontsize=14, fontweight="semibold")
fig.subplots_adjust(top=0.78, bottom=0.18)
plt.show()
plt.close(fig)

Now bind the two source traces. `merge_trace` does not collapse `COLOR` and `SHAPE`; it records how a new target assembly forms in `OBJECT` under their joint drive.

In [ ]:
object_before = Assembly(AREAS["object"], [])
object_trace = merge_trace(
    brain,
    AREAS["color"],
    AREAS["shape"],
    AREAS["object"],
    rounds=MERGE_ROUNDS,
)
object_assembly = object_trace.final

pd.DataFrame(object_trace.to_records())[
    ["round", "drive", "winner_count", "ever_fired", "new_winners", "overlap_prev"]
]

The merge animation is the target-area view: the sources are already active, and `OBJECT` settles into its own sparse trace.

In [ ]:
object_anim_fig, object_animation = animate_assembly_trace(
    object_trace,
    N,
    title="OBJECT merge dynamics",
    color=PALETTE["object"],
    interval=750,
)
object_animation_html = HTML(object_animation.to_jshtml())
plt.close(object_anim_fig)
object_animation_html

In [ ]:
object_metrics_fig, object_metrics_axes = plot_trace_metrics(
    object_trace,
    title="OBJECT merge stabilization",
    color=PALETTE["object"],
)
plt.show()
plt.close(object_metrics_fig)

The target is now inspectable as its own sparse trace. These maps are visual comparisons across areas; overlap claims should use assemblies from the same area.

In [ ]:
assembly_table(
    [
        ("source: COLOR", red_assembly),
        ("source: SHAPE", triangle_assembly),
        ("target: OBJECT", object_assembly),
    ],
    N,
)

In [ ]:
fig, axes = plot_assemblies(
    [red_assembly, triangle_assembly, object_assembly],
    n=N,
    titles=["source: COLOR", "source: SHAPE", "target: OBJECT"],
    subtitles=[
        f"{len(red_assembly)} winners | red cue",
        f"{len(triangle_assembly)} winners | triangle cue",
        f"{len(object_assembly)} winners | red + triangle",
    ],
    colors=[PALETTE["red"], PALETTE["triangle"], PALETTE["object"]],
    marker_size=18,
)
fig.suptitle(
    "Two source traces and the merged target trace",
    y=1.06,
    fontsize=14,
    fontweight="semibold",
)
fig.subplots_adjust(top=0.78, bottom=0.18)
plt.show()
plt.close(fig)

A stronger binding check asks whether either source alone can recover an overlapping `OBJECT` response. These probes use copied brains so they do not disturb the trace you just inspected.

In [ ]:
color_probe_brain = copy.deepcopy(brain)
shape_probe_brain = copy.deepcopy(brain)

color_response_trace = reciprocal_project_trace(
    color_probe_brain,
    AREAS["color"],
    AREAS["object"],
    rounds=SOURCE_RESPONSE_ROUNDS,
)
shape_response_trace = reciprocal_project_trace(
    shape_probe_brain,
    AREAS["shape"],
    AREAS["object"],
    rounds=SOURCE_RESPONSE_ROUNDS,
)

response_checks = pd.DataFrame(
    [
        {
            "probe": "COLOR alone -> OBJECT",
            "winner_count": len(color_response_trace.final),
            "overlap_with_merged_OBJECT": f"{overlap(object_assembly, color_response_trace.final):.2f}",
            "chance_baseline": f"{chance_overlap(K, N):.2%}",
        },
        {
            "probe": "SHAPE alone -> OBJECT",
            "winner_count": len(shape_response_trace.final),
            "overlap_with_merged_OBJECT": f"{overlap(object_assembly, shape_response_trace.final):.2f}",
            "chance_baseline": f"{chance_overlap(K, N):.2%}",
        },
    ]
)
response_checks

In [ ]:
response_fig, response_ax = plt.subplots(figsize=(6.8, 3.3))
plot_response_overlap(
    object_assembly,
    [color_response_trace.final, shape_response_trace.final],
    n=N,
    labels=["COLOR alone", "SHAPE alone"],
    ax=response_ax,
    color=PALETTE["object"],
    baseline_color=PALETTE["baseline"],
    title="Can either source recover the OBJECT trace?",
)
plt.show()
plt.close(response_fig)

What to notice:

- An assembly is sparse activity: `K` active winners out of `N` possible neurons.
- Projection is dynamical: the winner set turns over, then stabilizes.
- Merge creates a new target trace driven by two source traces.
- The best overlap diagnostics compare assemblies in the same area: `OBJECT` with `OBJECT`, not `COLOR` with `OBJECT`.
- The source-alone probes show what binding bought us in this toy setting: either source can drive an `OBJECT` response that overlaps the merged trace well above the random baseline.

Try next: change `K` from `80` to `40` or `120`, rerun the notebook, and watch the animations, turnover curves, and source-response overlaps move. Then continue to the readout notebook, where overlap becomes the bridge from traces to labels.